# Introduction
This notebook demonstrates how to set up and run a quantized version of the Llama-3-8B model. We will begin with some basic setup and then proceed to load and use the model.

## 1. Hello World

In [56]:
# First, let's print a simple message to ensure our environment is set up correctly.
print("Hello World")

Hello World


## 2. Initial Setup

In [1]:
#!cat /proc/meminfo | awk '/MemTotal/ {total=$2} /MemFree/ {free=$2} /MemAvailable/ {available=$2} END {printf "MemTotal: %.2f GB\nMemFree: %.2f GB\nMemAvailable: %.2f GB\n", total/1024/1024, free/1024/1024, available/1024/1024}'
#!nvidia-smi
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Free GPU Memory (GB): 34.8809


In [2]:
print("\n################################")
print("Setting up environment...")
print("################################\n")

import os
#os.chdir('..')
print("Current Working Directory ", os.getcwd())
import sys
sys.path.append("../") # Add directory containing src/data to path

import importlib
import src  # Assuming src is the package name

# Reload the src module after making changes
importlib.reload(src)

%load_ext autoreload
%autoreload 2

os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Disables parallelism to remove transformers warning

print("\n################################")
print("Setting up cache paths...")
print("################################\n")

os.environ["MKL_SERVICE_FORCE_INTEL"] = "1"
CACHE_PATH = "/nfs/students/daro/.cache/huggingface/"
print(f"Setting cache path to {CACHE_PATH}")

os.environ["TORCH_HOME"] = CACHE_PATH
os.environ["HF_HOME"] = CACHE_PATH
os.environ["HUGGINGFACE_HUB_CACHE"] = CACHE_PATH
os.environ["HUGGINGFACE_ASSETS_CACHE"] = CACHE_PATH

import torch
torch.hub.set_dir(CACHE_PATH)
with torch.no_grad():
    torch.cuda.empty_cache()
    
!cat /proc/meminfo | awk '/MemTotal/ {total=$2} /MemFree/ {free=$2} /MemAvailable/ {available=$2} END {printf "MemTotal: %.2f GB\nMemFree: %.2f GB\nMemAvailable: %.2f GB\n", total/1024/1024, free/1024/1024, available/1024/1024}'
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

# Code formatting and linting

# !black notebooks/Llama-3-8B-quant.ipynb
# !pylint notebooks/Llama-3-8B-quant.ipynb

print("\n################################")
print("Setting up cuda devices...")
print("################################\n")

if torch.cuda.is_available():
    print("CUDA device is available!")
    # Get the number of available CUDA devices
    num_cuda_devices = torch.cuda.device_count()
    print(f"Number of CUDA devices: {num_cuda_devices}")
    
    # Loop through available devices and get name
    for device_id in range(num_cuda_devices):
        device = torch.device(f"cuda:{device_id}")
        name = torch.cuda.get_device_name(device)
        print(f"  - CUDA Device {device_id+1}: {name}")
else:
    print("CUDA device is not available.")
    
print("\n################################")
print("Authentication with Hugging Face...")
print("################################\n")

import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
huggingface_token = os.getenv('HUGGINGFACE_TOKEN')

if huggingface_token is None:
    raise ValueError("Please set the HUGGINGFACE_TOKEN environment variable.")
else:
    print("Hugging Face token loaded successfully.")

login(token=huggingface_token, add_to_git_credential=True)
print("Successfully authenticated with the Hugging Face API.")

print("\n################################")
print("Setting up GPU memory usage list...")
print("################################\n")
# Global list to store GPU memory usage
from src.evaluations.evaluate_memory import record_gpu_memory
gpu_memory_usage = {}
record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Warm up notebook")


################################
Setting up environment...
################################

Current Working Directory  /nfs/homedirs/daro/git/quantization-reliability

################################
Setting up cache paths...
################################

Setting cache path to /nfs/students/daro/.cache/huggingface/
MemTotal: 1007.71 GB
MemFree: 180.03 GB
MemAvailable: 957.70 GB
Free GPU Memory (GB): 34.8809

################################
Setting up cuda devices...
################################

CUDA device is available!
Number of CUDA devices: 1
  - CUDA Device 1: NVIDIA A100-PCIE-40GB

################################
Authentication with Hugging Face...
################################

Hugging Face token loaded successfully.


/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Token is valid (permission: write).
Your token has been saved in your configured git credential helpers (store).
Your token has been saved to /nfs/students/daro/.cache/huggingface/token
Login successful
Successfully authenticated with the Hugging Face API.

################################
Setting up GPU memory usage list...
################################

Context: Warm up notebook. Free GPU Memory (GB): 34.8809


## 3. Loading Models

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

device = "cuda"

# model_name = "EleutherAI/gpt-neo-125m"  # Lightweight model for debugging purposes
# model_name = "meta-llama/Meta-Llama-3-8B"  # Too large to run on a gpu_gtx1080. GPU gpu_a100 is required.
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # Small enough to run on a gpu_gtx1080.
# model_name = "openai-community/gpt2-large"

tokenizer = AutoTokenizer.from_pretrained(model_name, device_map=device)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="cuda")
# TODO: Check why dtype = auto solved the problem
# TODO: what is the default value of torch_dtype -> look in the githubb documentation
# Always use "auto"
model.NAME = model_name

if tokenizer.model_max_length > 1e6:
  print(f"Tokenizer model max length reduced from {tokenizer.model_max_length} to 2048 to fit in memory")
  tokenizer.model_max_length = 2048

!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
print(f"Loaded model {model_name} with the following configuration:")
print(f"- model max length: {tokenizer.model_max_length}")
print(f"- dtype: {model.dtype}")
print(f"- device: {model.device}")
print(f"- parameters: {(lambda p: f'{p / 1e9:.1f}B' if p > 1e9 else (f'{p / 1e6:.1f}M' if p > 1e6 else str(p)))(model.num_parameters())}")
print(f"- memory footprint: {model.get_memory_footprint() / (1024 ** 3):.2f} GB")
print(f"- vocabulary size: {tokenizer.vocab_size}")
print(f"- padding token ID: {tokenizer.pad_token_id}")
print(f"- special tokens: {tokenizer.special_tokens_map}")

from src.evaluations.evaluate_memory import record_gpu_memory
record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Load model")

Free GPU Memory (GB): 30.3711
Loaded model TinyLlama/TinyLlama-1.1B-Chat-v1.0 with the following configuration:
- model max length: 2048
- dtype: torch.float32
- device: cuda:0
- parameters: 1.1B
- memory footprint: 4.10 GB
- vocabulary size: 32000
- padding token ID: 2
- special tokens: {'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '</s>'}
Context: Load model. Free GPU Memory (GB): 30.3711


In [54]:
# Example inference

from transformers import AutoTokenizer
import transformers 
import torch

tokenizer = AutoTokenizer.from_pretrained(model_name)
pipeline = transformers.pipeline(
    "text-generation",
    model=model_name,
    torch_dtype="auto",
    device_map="auto",
)

prompt = "What famous tower is in Paris?"
formatted_prompt = (
    f"### Human: {prompt}### Assistant:"
)

sequences = pipeline(
    formatted_prompt,
    do_sample=True,
    top_k=50,
    top_p = 0.7,
    num_return_sequences=1,
    repetition_penalty=1.1,
    max_new_tokens=500,
)
for seq in sequences:
    print(f"Result: {seq['generated_text']}")


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Loading checkpoint shards: 100%|██████████| 4/4 [00:10<00:00,  2.59s/it]
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


KeyboardInterrupt: 

In [ ]:
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

## 4. Loading Datasets

### 4.1. WikiText

In [4]:
# Initialize the datamodule
import os
from src.data.WikiTextDataModule import WikiTextDataModule

print("\n################################")
print("Setting up WikiTextDataModule...")
print("################################\n")

# wikitext_sequence_length = tokenizer.model_max_length
# wikitext_sequence_length = 10
wikitext_sequence_length = 2048
wikitext_batch_size = 1  # Just use batch size 1 for this project
wikitext_stride = 2048
wikitext_seed = 3
# wikitext_n_lines = 406
wikitext_n_lines = None

wikitext_data_module = WikiTextDataModule(
  directory_dataset=os.getcwd(),
  batch_size=wikitext_batch_size,
  sequence_length=wikitext_sequence_length,
  stride=wikitext_stride,
  tokenizer_name=model_name,
  seed=wikitext_seed,
  n_lines = wikitext_n_lines
)

wikitext_dataloader = wikitext_data_module.test_dataloader()

print("\n################################")
print("Printing properties of WikiTextDataModule...")
print("################################\n")

# Print properties
print(f"Length of train dataset: {len(wikitext_data_module.train_dataset)}")
print(f"Length of validation dataset: {len(wikitext_data_module.val_dataset)}")
print(f"Length of test dataset: {len(wikitext_data_module.test_dataset)}")

print("\nTotal number of tokens in each dataset:")
print(f"Train dataset: {sum([len(data_string) for data_string in wikitext_data_module.train_dataset['text']])}")
print(f"Validation dataset: {sum([len(data_string) for data_string in wikitext_data_module.val_dataset['text']])}")
print(f"Test dataset: {sum([len(data_string) for data_string in wikitext_data_module.test_dataset['text']])}")

total_string = "".join([data_string for data_string in wikitext_data_module.val_dataset['text']])
total_string_len = len(total_string)
tokenized_string = tokenizer.encode(total_string, return_tensors="pt")

print(f"\nLength of total validation dataset (characters): {total_string_len}")
print(f"Length of tokenized validation dataset (tokens): {len(tokenized_string[0])}")
print(f"Tokenizer compression rate: {(100 * len(tokenized_string[0]) / total_string_len):.2f}%")

# Reason why the numbers are low: number of tokens / 2048 -> gives the number of elements in the dataset
dataset_size = len(wikitext_dataloader)
print(f"\nNumber of batches in validation dataloader: {dataset_size}")

for i, (data, target) in enumerate(wikitext_dataloader):
    if i < 1:
        print(f"\nBatch {i + 1}:")
        original_text = tokenizer.decode(data[0], skip_special_tokens=True)
        print(f"  Original Text: {original_text[:500]}...")  # Print the first 500 characters
        print(f"  Input data (first 5 tokens): {data[0][:5]}")
        print(f"  Target labels (first 5 tokens): {target[0][:5]}")
        print(f"  Input data shape: {data.shape}")
        print(f"  Target labels shape: {target.shape}")


################################
Setting up WikiTextDataModule...
################################



Token indices sequence length is longer than the specified maximum sequence length for this model (341469 > 2048). Running this sequence through the model will result in indexing errors



################################
Printing properties of WikiTextDataModule...
################################

Length of train dataset: 36718
Length of validation dataset: 3760
Length of test dataset: 4358

Total number of tokens in each dataset:
Train dataset: 10892990
Validation dataset: 1142150
Test dataset: 1285622


Token indices sequence length is longer than the specified maximum sequence length for this model (292446 > 2048). Running this sequence through the model will result in indexing errors



Length of total validation dataset (characters): 1142150
Length of tokenized validation dataset (tokens): 292446
Tokenizer compression rate: 25.60%

Number of batches in validation dataloader: 166

Batch 1:
  Original Text: 

 = Robert Boulter = 




 Robert Boulter is an English film , television and theatre actor . He had a guest @-@ starring role on the television series The Bill in 2000 . This was followed by a starring role in the play Herons written by Simon Stephens , which was performed in 2001 at the Royal Court Theatre . He had a guest role in the television series Judge John Deed in 2002 . In 2004 Boulter landed a role as " Craig " in the episode " Teddy 's Story " of the television series The Long Firm ...
  Input data (first 5 tokens): tensor([    1, 29871,    13,    13,   353])
  Target labels (first 5 tokens): tensor([29871,    13,    13,   353,  4755])
  Input data shape: torch.Size([1, 2048])
  Target labels shape: torch.Size([1, 2048])


In [42]:
from tqdm import tqdm

from datasets import load_dataset
test = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test')
encodings = tokenizer('\n\n'.join(test['text']), return_tensors='pt')

max_length = tokenizer.model_max_length
stride = 1024

lls = []
input_ids_list = []
target_ids_list = []
for i in tqdm(range(0, encodings.input_ids.size(1), stride)):
    # if i > stride:
    #     break
    begin_loc = max(i + stride - max_length, 0)
    end_loc = i + stride
    input_ids = encodings.input_ids[:,begin_loc:end_loc].to(device)
    target_ids = input_ids.clone()
    target_ids[:,:-stride] = -100
    input_ids_list.append(input_ids)
    target_ids_list.append(target_ids)

100%|██████████| 334/334 [00:00<00:00, 18262.25it/s]


In [44]:
print(input_ids_list[0].shape)
print(len(input_ids_list))
print(len(wikitext_dataloader))

print(tokenizer.decode(input_ids_list[0][0][:50], skip_special_tokens=True))
# print(encodings.input_ids[:, :50])
print(tokenizer.decode(encodings.input_ids[:, :50][0], skip_special_tokens=True))

for i, (x, y) in enumerate(wikitext_dataloader):
    if i < 1:
        print(tokenizer.decode(x[0][:50], skip_special_tokens=True))
        print(tokenizer.decode(y[0][:50], skip_special_tokens=True))

torch.Size([1, 1024])
334
333


 = Robert Boulter = 




 Robert Boulter is an English film , television and theatre actor . He had a guest @-@ starring role on the television series The Bill in 20


 = Robert Boulter = 




 Robert Boulter is an English film , television and theatre actor . He had a guest @-@ starring role on the television series The Bill in 20


 = Robert Boulter = 




 Robert Boulter is an English film , television and theatre actor . He had a guest @-@ starring role on the television series The Bill in 20


 = Robert Boulter = 




 Robert Boulter is an English film , television and theatre actor . He had a guest @-@ starring role on the television series The Bill in 200


In [14]:
dataloader = wikitext_data_module.val_dataloader()
print(dataloader.dataset.dataset.shape)

(406, 1)


In [ ]:
decoded = tokenizer.decode(encoded['input_ids'])
print(decoded)

### 4.2. OpenAssistant

In [ ]:
# Initialize the datamodule
import os
from src.data.OpenAssistantDataModule import OpenAssistantDataModule

directory_dataset = os.getcwd()
oasst_batch_size = 1  # Just use batch size 1 for this project
# oasst_batch_size = 16
# oasst_batch_size = 64
oasst_sequence_length = 512  # Maximum sequence length - use the default value
oasst_seed = 1

# Data Module
oasst_data_module = OpenAssistantDataModule(
  directory_dataset=directory_dataset,
  batch_size=oasst_batch_size,
  sequence_length=oasst_sequence_length,
  tokenizer_name=model_name,
  seed=oasst_seed
)

# Data Loader
# oasst_dataloader = oasst_data_module.train_dataloader()
oasst_dataloader = oasst_data_module.val_dataloader()

print(f"Length of datasets:", len(oasst_data_module.train_dataset), len(oasst_data_module.val_dataset))

# Reason why the numbers are low: number of tokens / 2048 -> gives the number of elements in the dataset

oasst_dataset_size = len(oasst_dataloader)
print(f"Number of batches in train_dataloader: {dataset_size}")
for i, (data, target) in enumerate(oasst_dataloader):
    if i < 2:
        print(f"Batch {i+1}:")
        original_text = tokenizer.decode(data[0], skip_special_tokens=True)
        print(f"  Original Text: {original_text[:500]}")
        print(f"  Input data (first 5 tokens): {data[0][:5]}")
        print(f"  Target labels (first 5 tokens): {target[0][:5]}")
        print(f"  Input data shape: {data.shape}")
        print(f"  Target labels shape: {target.shape}")

## 5. Quantization

### 5.1. BitsAndBytes

#### 5.1.1 BitsAndBytes 8-bit

In [ ]:
# BNB Config 8-bit

from transformers import BitsAndBytesConfig

from src import MODEL_SAVE_PATH

# Define the device and model name
device = "cuda"
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# BnB Quantization Configurations
bnb_config_8bit = BitsAndBytesConfig(
    load_in_8bit=True,
    load_in_4bit=False,
    llm_int8_threshold=6.0,
    llm_int8_enable_fp32_cpu_offload=False,
    llm_int8_has_fp16_weight=False,
)

# Save path
import os
bnb_8bit_model_name = f"{model_name.split('/')[1]}-bnb-8bit"
bnb_8bit_model_path = os.path.join(MODEL_SAVE_PATH, bnb_8bit_model_name)
os.makedirs(bnb_8bit_model_path, exist_ok=True)

In [ ]:
# Quantization 8-bit

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name, device_map=device)
model_bnb_8bit = AutoModelForCausalLM.from_pretrained(
    model_name, 
    quantization_config=bnb_config_8bit, 
    torch_dtype="auto",
    device_map=device
)
model_bnb_8bit.NAME = bnb_8bit_model_name

print(f"8-bit BNB Model Memory Footprint: {(model_bnb_8bit.get_memory_footprint() / (1024 ** 3)):.2f} GB")

from accelerate import Accelerator
accelerate = Accelerator()
accelerate.save_model(model_bnb_8bit, bnb_8bit_model_path)

print(f"8-bit BnB model saved at: {bnb_8bit_model_path}")

# from src.models.utils_llm import calculate_model_size
# calculate_model_size(bnb_8bit_model_path)
# from src.models.utils_llm import print_gpu_utilization
# print_gpu_utilization()

#### 5.1.2 BitsAndBytes 4-bit

In [ ]:
# BNB Config 4-bit

import torch
from transformers import BitsAndBytesConfig

from src import MODEL_SAVE_PATH

# Define the device and model name
device = "cuda"
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config_4bit = BitsAndBytesConfig(
    load_in_8bit=False,
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="fp4",
    bnb_4bit_use_double_quant=False,
)

# Save path
import os
bnb_4bit_model_name = f"{model_name.split('/')[1]}-bnb-4bit"
bnb_4bit_model_path = os.path.join(MODEL_SAVE_PATH, bnb_4bit_model_name)

In [ ]:
# Quantization 4-bit

from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name, device_map=device)
model_bnb_4bit = AutoModelForCausalLM.from_pretrained(
    model_name, 
    quantization_config=bnb_config_4bit, 
    torch_dtype="auto",
    device_map=device
)
model_bnb_4bit.NAME = bnb_4bit_model_name

print(f"4-bit BNB Model Memory Footprint: {(model_bnb_4bit.get_memory_footprint() / (1024 ** 3)):.2f} GB")

from accelerate import Accelerator
accelerate = Accelerator()
accelerate.save_model(model_bnb_4bit, bnb_4bit_model_path)

print(f"4-bit BnB model saved at: {bnb_4bit_model_path}")

# from src.models.utils_llm import calculate_model_size
# calculate_model_size(bnb_4bit_model_path)
# from src.models.utils_llm import print_gpu_utilization
# print_gpu_utilization()

### 5.2 AWQ

In [ ]:
# AWQ Config
awq_config = {
    "zero_point": True,
    "q_group_size": 128,
    "w_bit": 4,
    "version": "GEMM"
}

# AWQ Calibration Split
awq_calib_split = "validation"

# Save path
import os
awq_model_name = f"{model_name.split('/')[1]}-awq"
awq_model_path = os.path.join(MODEL_SAVE_PATH, awq_model_name)

# Define the device and model name
device = "cuda"
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

In [ ]:
# AWQ Quantization

from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer

awq_tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
awq_model = AutoAWQForCausalLM.from_pretrained(
    model_name,
    device_map=device
)

# Quantize with wikitext validation as calibration data
awq_model.quantize(
    tokenizer=tokenizer,
    quant_config=awq_config,
    calib_data=wikitext_dataset,  # Pass the loaded validation dataset here
)
awq_model.NAME = awq_model_name

# Save quantized model
awq_model.save_quantized(awq_model_path)
awq_tokenizer.save_pretrained(awq_model_path)

print(f'Model is quantized and saved at "{awq_model_path}"')

# from src.models.utils_llm import calculate_model_size
# calculate_model_size(awq_model_path)
# from src.models.utils_llm import print_gpu_utilization
# print_gpu_utilization()

In [ ]:
# Load model and generate text
awq_model_path = "TinyLlama-1.1B-Chat-v1.0-awq"

awq_tokenizer = AutoTokenizer.from_pretrained(awq_model_path)
awq_model = AutoAWQForCausalLM.from_pretrained(
    awq_model_path,
    trust_remote_code=True,
)

# Generate text
prompt = "What is the weather like today?"
input_ids = awq_tokenizer.encode(prompt, return_tensors="pt")  # Convert text to tensors

output = awq_model.generate(input_ids)
generated_text = awq_tokenizer.decode(output[0], skip_special_tokens=True)

### 5.3 HQQ

In [11]:
# HQQ Config

from transformers import AutoTokenizer

# Define the model name and tokenizer
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v0.1"
tokenizer = AutoTokenizer.from_pretrained(model_name)

### 5.3.1 Option 1: All linear layers will use the same quantization config

In [12]:
import torch
from transformers import AutoModelForCausalLM, HqqConfig
from accelerate import Accelerator

# Option 1: All linear layers will use the same quantization config
quant_config_same = HqqConfig(
    nbits=8, 
    group_size=64, 
    quant_zero=False, 
    quant_scale=False, 
    axis=0  # Default value
)

# Quantize the model with the same config for all linear layers
model_same = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="cuda",
    quantization_config=quant_config_same
)

# Saving the model is not possible due to the current version of transformers
# ValueError: The model is quantized with hqq and is not serializable -
# check out the warnings from the logger on the traceback to understand the reason why the quantized model is not serializable.
# ValueError: .to is not supported for HQQ-quantized models.

### 5.3.2. Option 2: Different configs for specific layers

In [13]:
import torch
from transformers import AutoModelForCausalLM, HqqConfig

# Option 2: Different configs for specific layers
q4_config = {'nbits': 4, 'group_size': 64, 'quant_zero': False, 'quant_scale': False}
q3_config = {'nbits': 3, 'group_size': 32, 'quant_zero': False, 'quant_scale': False}

quant_config_dynamic = HqqConfig(dynamic_config={
    'self_attn.q_proj': q4_config,
    'self_attn.k_proj': q4_config,
    'self_attn.v_proj': q4_config,
    'self_attn.o_proj': q4_config,
    'mlp.gate_proj': q3_config,
    'mlp.up_proj': q3_config,
    'mlp.down_proj': q3_config,
})

# Quantize the model with different configs for specific layers
model_dynamic = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="cuda",
    quantization_config=quant_config_dynamic
)

### 5.4 Quanto

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, QuantoConfig

device = "cuda"
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v0.1"
tokenizer = AutoTokenizer.from_pretrained(model_name)

quantization_config = QuantoConfig(
  weights="int8"
)
quantized_model = AutoModelForCausalLM.from_pretrained(model_name, device_map=device, quantization_config=quantization_config)

## 6. Evaluation

In [11]:
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Free GPU Memory (GB): 34.7051


### 6.1. Perplexity

In [7]:
import torch
import torchmetrics
import tqdm
from torch.cuda.amp import autocast

# Evaluate Perplexity
print("\n################################")
print("Evaluating Perplexity...")
print("################################\n")

def evaluate_perplexity(model, dataloader, device="cuda", to_device=False):
    if isinstance(model, torch.nn.Module):
        model.eval()
        print(f"Model in evaluation mode. Device: {device}")
    metric = torchmetrics.text.Perplexity(ignore_index=-100).to(device)  # -100 is the padding token.

    for i, (x, y) in enumerate(dataloader):
        print(f"Processing batch {i}")
        x, y = x.to(device), y.to(device)
        
        with torch.no_grad():
            outputs = model(x)
            logits = outputs.logits
            !nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
            
            # Metric on current batch
            perplexity = metric(logits.float(), y)
            print(f"Perplexity: {perplexity:.2f}")

    # Metric on all batches using custom accumulation
    perplexity = metric.compute()
    print(f"\nFinal Perplexity (PPL): {perplexity:.3f}")
    return perplexity.item()


################################
Evaluating Perplexity...
################################



In [8]:
wikitext_dataloader = wikitext_data_module.test_dataloader()
ppl = evaluate_perplexity(model, wikitext_dataloader, device=device)
print(f"\nFinal Perplexity (PPL): {ppl:.3f}")

Model in evaluation mode. Device: cuda
Processing batch 0
Free GPU Memory (GB): 27.6621
Perplexity: 7.59
Processing batch 1
Free GPU Memory (GB): 27.416
Perplexity: 12.65
Processing batch 2
Free GPU Memory (GB): 27.416
Perplexity: 16.74
Processing batch 3
Free GPU Memory (GB): 27.416
Perplexity: 16.14
Processing batch 4
Free GPU Memory (GB): 27.416
Perplexity: 8.50
Processing batch 5
Free GPU Memory (GB): 27.416
Perplexity: 10.69
Processing batch 6
Free GPU Memory (GB): 27.416
Perplexity: 7.87
Processing batch 7
Free GPU Memory (GB): 27.416
Perplexity: 7.29
Processing batch 8
Free GPU Memory (GB): 27.416
Perplexity: 11.59
Processing batch 9
Free GPU Memory (GB): 27.416
Perplexity: 13.24
Processing batch 10
Free GPU Memory (GB): 27.416
Perplexity: 11.91
Processing batch 11
Free GPU Memory (GB): 27.416
Perplexity: 10.59
Processing batch 12
Free GPU Memory (GB): 27.416
Perplexity: 9.18
Processing batch 13
Free GPU Memory (GB): 27.416
Perplexity: 11.22
Processing batch 14
Free GPU Memory (

In [14]:
wikitext_dataloader = wikitext_data_module.test_dataloader()
ppl = evaluate_perplexity(model_same, wikitext_dataloader, device=device)
print(f"\nFinal Perplexity (PPL): {ppl:.3f}")

Model in evaluation mode. Device: cuda
Processing batch 0
Free GPU Memory (GB): 26.7715
Perplexity: 7.59
Processing batch 1
Free GPU Memory (GB): 26.5254
Perplexity: 12.65
Processing batch 2
Free GPU Memory (GB): 26.5254
Perplexity: 16.76
Processing batch 3
Free GPU Memory (GB): 26.5254
Perplexity: 16.14
Processing batch 4
Free GPU Memory (GB): 26.5254
Perplexity: 8.49
Processing batch 5
Free GPU Memory (GB): 26.5254
Perplexity: 10.68
Processing batch 6
Free GPU Memory (GB): 26.5254
Perplexity: 7.87
Processing batch 7
Free GPU Memory (GB): 26.5254
Perplexity: 7.28
Processing batch 8
Free GPU Memory (GB): 26.5254
Perplexity: 11.58
Processing batch 9
Free GPU Memory (GB): 26.5254
Perplexity: 13.24
Processing batch 10
Free GPU Memory (GB): 26.5254
Perplexity: 11.92
Processing batch 11
Free GPU Memory (GB): 26.5254
Perplexity: 10.58
Processing batch 12
Free GPU Memory (GB): 26.5254
Perplexity: 9.18
Processing batch 13
Free GPU Memory (GB): 26.5254
Perplexity: 11.21
Processing batch 14
Free

In [15]:
wikitext_dataloader = wikitext_data_module.test_dataloader()
ppl = evaluate_perplexity(model_dynamic, wikitext_dataloader, device=device)
print(f"\nFinal Perplexity (PPL): {ppl:.3f}")

Model in evaluation mode. Device: cuda
Processing batch 0
Free GPU Memory (GB): 26.5254
Perplexity: 7.92
Processing batch 1
Free GPU Memory (GB): 26.5254
Perplexity: 13.04
Processing batch 2
Free GPU Memory (GB): 26.5254
Perplexity: 17.44
Processing batch 3
Free GPU Memory (GB): 26.5254
Perplexity: 16.92
Processing batch 4
Free GPU Memory (GB): 26.5254
Perplexity: 8.75
Processing batch 5
Free GPU Memory (GB): 26.5254
Perplexity: 11.02
Processing batch 6
Free GPU Memory (GB): 26.5254
Perplexity: 8.07
Processing batch 7
Free GPU Memory (GB): 26.5254
Perplexity: 7.53
Processing batch 8
Free GPU Memory (GB): 26.5254
Perplexity: 12.01
Processing batch 9
Free GPU Memory (GB): 26.5254
Perplexity: 13.60
Processing batch 10
Free GPU Memory (GB): 26.5254
Perplexity: 12.79
Processing batch 11
Free GPU Memory (GB): 26.5254
Perplexity: 11.07
Processing batch 12
Free GPU Memory (GB): 26.5254
Perplexity: 9.73
Processing batch 13
Free GPU Memory (GB): 26.5254
Perplexity: 11.60
Processing batch 14
Free

In [9]:
import numpy as np
lls = torch.tensor(lls)
print(stride)
print(lls/stride)
print(torch.exp(lls / (stride)))
print(torch.exp(lls.sum() / (31 * stride)))

ppls = [ppl for ppl in ppls]
print(ppls)

print(xs[2])
print(ys[2])
print(input_ids_list[2])
print(target_ids_list[2])

print(outputs[0])
print()

1024
tensor([2.4563, 3.2453, 3.0608, 3.2338, 3.4153, 3.2130, 3.3681, 3.1029, 2.8991,
        3.0156, 2.8298])
tensor([11.6612, 25.6692, 21.3436, 25.3749, 30.4268, 24.8541, 29.0234, 22.2628,
        18.1581, 20.4011, 16.9424])
tensor(2.9791)
[tensor(10.4680, device='cuda:0', grad_fn=<SqueezeBackward0>), tensor(20.8812, device='cuda:0', grad_fn=<SqueezeBackward0>), tensor(20.0631, device='cuda:0', grad_fn=<SqueezeBackward0>)]
tensor([[4054,  837,  284,  ...,  262, 3931,  286]], device='cuda:0')
tensor([[ 837,  284,  465,  ..., 3931,  286,  767]], device='cuda:0')
tensor([[4054,  837,  284,  ...,  262, 3931,  286]], device='cuda:0')
tensor([[4054,  837,  284,  ...,  262, 3931,  286]], device='cuda:0')
tensor(2.8298, device='cuda:0')


/tmp/ipykernel_2523226/3573369581.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lls = torch.tensor(lls)


In [5]:
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Free GPU Memory (GB): 8.41992


In [ ]:
evaluate_perplexity(model_bnb_8bit, tokenizer, wikitext_data_module, device=device)

In [ ]:
evaluate_perplexity(model_bnb_4bit, tokenizer, wikitext_data_module, device=device)

In [ ]:
evaluate_perplexity(awq_model, wikitext_dataloader, device="cuda")

In [ ]:
list_of_models = [model, model_bnb_8bit, model_bnb_4bit]
results = {}
for model in list_of_models:
  print(f"Perplexity for model {model.NAME}: {evaluate_perplexity(model_bnb_8bit, wikitext_data_module, device)}"

# Print perplexity results
#print(f"Perplexity (8-bit): {perplexity_8bit:.4f}")
#print(f"Perplexity (4-bit): {perplexity_4bit:.4f}")
print(f"Perplexity (Original): {perplexity_original:.4f}")

### 6.2. Brier Score

In [4]:
import logging
from torch.cuda.amp import autocast
import torch.nn.functional as F
import torch

def evaluate_brier_score(model, dataloader, device="cuda", to_device=False):
    if to_device:
        model.to(device)
    if isinstance(model, torch.nn.Module):
        model.eval()
        logging.info(f"Model in evaluation mode. Device: {device}")
        
    brier_sum = 0

    for i, (x, y) in enumerate(dataloader):
        if i > 10:
            break
        print(f"Processing batch {i}")
        x, y = x.to(device), y.to(device)
        
        with torch.no_grad() and autocast():
            outputs = model(x)
            logits = outputs.logits
            !nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
            
            # Shift logits and target_ids to the left by 1 for calculating the Brier score
            shifted_logits = logits[:, :-1].contiguous()
            shifted_target_ids = x[:, 1:].contiguous()

            # Flatten the logits and target_ids for calculation
            shifted_logits = shifted_logits.view(-1, shifted_logits.size(-1))
            shifted_target_ids = shifted_target_ids.view(-1)

            # Filter out the -100 targets
            valid_indices = shifted_target_ids != -100
            valid_logits = shifted_logits[valid_indices]
            valid_target_ids = shifted_target_ids[valid_indices]

            # Get the probabilities
            probs = F.softmax(valid_logits, dim=-1)

            # Create one-hot target vectors
            targets = F.one_hot(valid_target_ids, num_classes=probs.size(-1)).float()

            # Calculate the Brier score
            brier_score = torch.mean((probs - targets) ** 2)
            print(f"Brier Score: {brier_score:.4f}")
            brier_sum += brier_score
            del brier_score

    avg_brier_score = brier_sum / len(dataloader)
    print(f"Final Brier Score: {avg_brier_score:.4f}")
    
    return avg_brier_score

wikitext_dataloader = wikitext_data_module.test_dataloader()
evaluate_brier_score(model, wikitext_dataloader, device=device)

Processing batch 0
Free GPU Memory (GB): 34.5293
Brier Score: 0.0000
Processing batch 1
Free GPU Memory (GB): 31.8242
Brier Score: 0.0000
Processing batch 2
Free GPU Memory (GB): 29.1309
Brier Score: 0.0000
Processing batch 3
Free GPU Memory (GB): 26.7988
Brier Score: 0.0000
Processing batch 4
Free GPU Memory (GB): 24.4746
Brier Score: 0.0000
Processing batch 5
Free GPU Memory (GB): 22.1406
Brier Score: 0.0000
Processing batch 6
Free GPU Memory (GB): 19.8164
Brier Score: 0.0000
Processing batch 7
Free GPU Memory (GB): 17.4844
Brier Score: 0.0000
Processing batch 8
Free GPU Memory (GB): 15.1602
Brier Score: 0.0000
Processing batch 9
Free GPU Memory (GB): 12.8281
Brier Score: 0.0000
Processing batch 10
Free GPU Memory (GB): 10.502
Brier Score: 0.0000
Final Brier Score: 0.0000


tensor(6.6241e-07, device='cuda:0', grad_fn=<DivBackward0>)

In [7]:
import logging
from torch.cuda.amp import autocast
import torch.nn.functional as F
import torch
import torchmetrics

def evaluate_brier_score(model, dataloader, device="cuda", to_device=False):
    if to_device:
        model.to(device)
    if isinstance(model, torch.nn.Module):
        model.eval()
    logging.info(f"Model in evaluation mode. Device: {device}")

    # Initialize the MeanSquaredError metric
    metric = torchmetrics.MeanSquaredError().to(device)

    for i, (x, y) in enumerate(dataloader):
        if i > 10:
            break
        print(f"Processing batch {i}")
        x, y = x.to(device), y.to(device)
        with torch.no_grad() and autocast():
            outputs = model(x)
            logits = outputs.logits
            
        !nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

        # Shift logits and target_ids to the left by 1 for calculating the Brier score
        shifted_logits = logits[:, :-1].contiguous()
        shifted_target_ids = x[:, 1:].contiguous()

        # Flatten the logits and target_ids for calculation
        shifted_logits = shifted_logits.view(-1, shifted_logits.size(-1))
        shifted_target_ids = shifted_target_ids.view(-1)

        # Filter out the -100 targets
        valid_indices = shifted_target_ids != -100
        valid_logits = shifted_logits[valid_indices]
        valid_target_ids = shifted_target_ids[valid_indices]

        # Get the probabilities
        probs = F.softmax(valid_logits, dim=-1)

        # Create one-hot target vectors
        targets = F.one_hot(valid_target_ids, num_classes=probs.size(-1)).float()

        # Update the metric
        metric.update(probs, targets)

        # Calculate and print the current Brier score
        current_brier_score = metric.compute()
        print(f"Current Brier Score: {current_brier_score:.4f}")

    # Compute the final Brier score
    final_brier_score = metric.compute()
    print(f"Final Brier Score: {final_brier_score:.10f}")

    return final_brier_score.item()

wikitext_dataloader = wikitext_data_module.test_dataloader()
brier_score = evaluate_brier_score(model, wikitext_dataloader, device=device)
print(f"\nFinal Brier Score: {brier_score:.10f}")

# Print GPU memory usage
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Processing batch 0
Free GPU Memory (GB): 28.9004
Current Brier Score: 0.0000
Processing batch 1
Free GPU Memory (GB): 28.9004
Current Brier Score: 0.0000
Processing batch 2
Free GPU Memory (GB): 28.9004
Current Brier Score: 0.0000
Processing batch 3
Free GPU Memory (GB): 28.9004
Current Brier Score: 0.0000
Processing batch 4
Free GPU Memory (GB): 28.9004
Current Brier Score: 0.0000
Processing batch 5
Free GPU Memory (GB): 28.9004
Current Brier Score: 0.0000
Processing batch 6
Free GPU Memory (GB): 28.9004
Current Brier Score: 0.0000
Processing batch 7
Free GPU Memory (GB): 28.9004
Current Brier Score: 0.0000
Processing batch 8
Free GPU Memory (GB): 28.9004
Current Brier Score: 0.0000
Processing batch 9
Free GPU Memory (GB): 28.9004
Current Brier Score: 0.0000
Processing batch 10
Free GPU Memory (GB): 28.9004
Current Brier Score: 0.0000
Final Brier Score: 0.0000200531

Final Brier Score: 0.0000200531
Free GPU Memory (GB): 28.9004


In [ ]:
import torch
import torch.nn.functional as F
from torch.cuda.amp import autocast

class BrierScore:
    def __init__(self, device="cpu"):
        self.device = device
        self.reset()

    def reset(self):
        self.total_brier_score = 0.0
        self.num_batches = 0

    def update(self, probs, targets):
        brier_score = torch.mean((probs - targets) ** 2)
        self.total_brier_score += brier_score.item()
        self.num_batches += 1

    def compute(self):
        if self.num_batches == 0:
            return 0.0
        return self.total_brier_score / self.num_batches

def evaluate_brier_score(model, dataloader, device="cuda", to_device=False):
    if to_device:
        model.to(device)

    if isinstance(model, torch.nn.Module):
        model.eval()

    print(f"Model in evaluation mode. Device: {device}")
    
    # Initialize BrierScore metric
    metric = BrierScore(device=device)
    
    for i, (x, y) in enumerate(dataloader):
        if i > 10:
            break
        print(f"Processing batch {i}")
        x, y = x.to(device), y.to(device)

        with torch.no_grad() and autocast():
            outputs = model(x)
            logits = outputs.logits
            !nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

            # Shift logits and target_ids to the left by 1 for calculating the Brier score
            shifted_logits = logits[:, :-1].contiguous()
            shifted_target_ids = x[:, 1:].contiguous()

            # Flatten the logits and target_ids for calculation
            shifted_logits = shifted_logits.view(-1, shifted_logits.size(-1))
            shifted_target_ids = shifted_target_ids.view(-1)

            # Filter out the -100 targets
            valid_indices = shifted_target_ids != -100
            valid_logits = shifted_logits[valid_indices]
            valid_target_ids = shifted_target_ids[valid_indices]

            # Get the probabilities
            probs = F.softmax(valid_logits, dim=-1)

            # Create one-hot target vectors
            targets = F.one_hot(valid_target_ids, num_classes=probs.size(-1)).float()

            # Update the metric with the current batch's results
            metric.update(probs, targets)

    # Compute the final Brier score across all batches
    avg_brier_score = metric.compute()
    print(f"Final Brier Score: {avg_brier_score:.10f}")

    return avg_brier_score

# Assuming wikitext_data_module and model are defined elsewhere
wikitext_dataloader = wikitext_data_module.test_dataloader()
final_brier_score = evaluate_brier_score(model, wikitext_dataloader, device=device)
print(f"\nFinal Brier Score: {final_brier_score:.10f}")

In [13]:
evaluate_brier_score(model, tokenizer, wikitext_dataloader, factor=100, device=device)

100%|██████████| 6/6 [00:01<00:00,  5.99it/s]

Brier Score of model TinyLlama/TinyLlama-1.1B-Chat-v0.1: 0.0000


tensor(1.8003e-05, device='cuda:0')

In [14]:
evaluate_brier_score(model_bnb_4bit, tokenizer, wikitext_data_module, device=device)

NameError: name 'model_bnb_4bit' is not defined

In [ ]:
evaluate_brier_score(model_bnb_8bit, tokenizer, wikitext_data_module, device=device)

In [ ]:
evaluate_brier_score(awq_model, tokenizer, wikitext_data_module, device=device)

# 7. SEML Pipeline

In [2]:
import shutil
import re
import os
import torch

# To avoid the following problem when running seml (see https://github.com/pytorch/pytorch/issues/37377)
os.environ["MKL_SERVICE_FORCE_INTEL"] = "1"
if re.match(".*username.*", os.getcwd()):
    CACHE_PATH = "~/.cache/"
else:
    CACHE_PATH = "/tmp/"

torch.hub.set_dir(CACHE_PATH)

import logging
logger = logging.getLogger("quant_logger")

#os.chdir('..')
print("Current Working Directory " , os.getcwd())
import sys
sys.path.append("../") # Add directory containing src/data to path

import importlib
import src  # Assuming src is the package name
import seml

# Reload the src module after making changes
importlib.reload(src)

from src.models import get_model, get_model_name
from src.data import get_dataset, data_loader_from_split
from src.algorithms.quantization.quantize import quantize
from src.evaluations.evaluate_all import evaluate

from src.evaluations.evaluate_memory import record_gpu_memory
gpu_memory_usage = {}

Current Working Directory  /nfs/homedirs/daro/git/quantization-reliability


In [3]:
def run_quantize(
    # Dataset parameters
    seed_dataset=123,
    directory_dataset="",
    calib_dataset_name="",
    calib_dataset_split="",
    eval_dataset_name="",
    eval_dataset_split="",
    batch_size=1,
    dataset_stride=1024,
    dataset_seq_length=1024,
    # Model parameters
    seed_model=123,
    directory_model="",
    clean_cache=True,
    model_name="",
    # Quantization parameters
    quantize_method="",
    quantize_params={},
    # Evaluation metrics
    eval_metrics=[
        "perplexity",
    ],
    device="cuda",
    save_quantized_model=False,
    quantized_model_save_path="",
):
    ##################
    ## Print config ##
    ##################
    logger.info("Received the following configuration:")
    logger.info(
        f"Calibration dataset: {calib_dataset_name}\n"
        f"Calibration split: {calib_dataset_split}\n"
        f"Evaluation dataset: {eval_dataset_name}\n"
        f"Evaluation split: {eval_dataset_split}\n"
        f"Dataloader stride: {dataset_stride}\n"
        f"Dataloader sequence length: {dataset_seq_length}\n"
        f"Batch size: {batch_size}\n"
        f"Model: {model_name}\n"
        f"Quantize method: {quantize_method}\n"
        f"Quantize params: {quantize_params}\n"
        f"Evaluation metrics: {eval_metrics}\n"
        f"Device: {device}\n"
    )
    
    ################
    ## Load model ##
    ################
    logger.info("Load base model")
    model_full_name = get_model_name(model_name)
    model, tokenizer = get_model(
        model_name=model_full_name,
        seed=seed_model,
        directory_model=directory_model,
        device=device,
    )
    record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Load model")
    
    ###############
    ## Load data ##
    ###############
    logger.info("Load calibration and evaluation data modules")
    calib_data_module = get_dataset(
        dataset_name=calib_dataset_name,
        directory_dataset=directory_dataset,
        batch_size=batch_size,
        sequence_length=dataset_stride,
        tokenizer_name=model_full_name,
        seed=seed_dataset,
    )
    eval_data_module = get_dataset(
        dataset_name=eval_dataset_name,
        directory_dataset=directory_dataset,
        batch_size=batch_size,
        sequence_length=dataset_stride,
        tokenizer_name=model_full_name,
        seed=seed_dataset,
    )
    
    calib_dataloader = data_loader_from_split(calib_data_module)[calib_dataset_split]
    eval_dataloader = data_loader_from_split(eval_data_module)[eval_dataset_split]
    record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Load data")

    ################################
    ## Update quantize parameters ##
    ################################
    logger.info("Defining quantize parameters")
    quantize_params.update(quantize_params)
    logger.info(f"Default parameters adjusted from {quantize_params}")

    ##############
    ## Quantize ##
    ##############
    logger.info("Quantization")
    quantized_model = quantize(
        model_name=model_full_name,
        tokenizer=tokenizer,
        calib_dataloader=calib_dataloader,
        quantize_method=quantize_method,
        quantize_config=quantize_params,
        save_model=save_quantized_model,
        save_path=quantized_model_save_path,
        device=device
    )
    record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Quantize model")

    ##############
    ## Evaluate ##
    ##############
    logger.info("Evaluating the quantized models")
    results = evaluate(
        model=quantized_model,
        eval_dataloader=eval_dataloader,
        eval_metrics=eval_metrics,
        factor=100,
        device=device,
        to_device=(quantize_method in ["AWQ"]),
        prefix=f"{model_name}_",
    )
    record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Evaluate model")

    ####################
    ## Cleaning cache ##
    ####################
    logger.info
    if clean_cache:
        for root, dirs, files in os.walk(CACHE_PATH, topdown=False):
            for dir_name in dirs:
                pattern = re.compile(f"^.*{model_full_name.split('/')[-1]}.*")
                dir_path = os.path.join(root, dir_name)
                if re.match(pattern, dir_path):
                    try:
                        shutil.rmtree(dir_path)
                    except:
                        pass

    fail_trace = {
        "fail_trace": seml.evaluation.get_results,
    }

    return {**results, **fail_trace}


In [4]:
import itertools
import random
import torch  # Ensure torch is imported

# Fixed parameters
fixed_params = {
    'device': 'cuda',
    'clean_cache': True,
    'save_quantized_model': True,
    'seed_model': 123,
    'seed_dataset': 123,
    'batch_size': 1,
    'dataset_stride': 1024,
    'dataset_seq_length': 1024,
    'eval_metrics': ['perplexity', 'brier_score'],
    'calib_dataset_split': 'validation',
    'eval_dataset_split': 'test',
}

# Grid parameters
grid_params = {
    'calib_dataset_name': ['WikiText', 'OpenAssistant'],
    'eval_dataset_name': ['WikiText', 'OpenAssistant'],
    'quantize_method': ['BNB'],
    'quantize_params': [
        {
            "num_bits": 8,
            "llm_int8_threshold": 6.0,
            "llm_int8_enable_fp32_cpu_offload": False,
            "llm_int8_has_fp16_weight": False
        },
        {
            "num_bits": 4,
            "bnb_4bit_compute_dtype": torch.bfloat16,
            "bnb_4bit_quant_type": "fp4",
            "bnb_4bit_use_double_quant": False,
        }
    ],
    'model_name': ['TinyLlama']
}

batch_sizes = [1]

# Generate all combinations for grid search
grid_combinations = list(itertools.product(
    grid_params['calib_dataset_name'],
    grid_params['eval_dataset_name'],
    grid_params['quantize_method'],
    grid_params['quantize_params'],
    grid_params['model_name']
))

# Run the quantize function for all combinations
results = []
max_combinations = 10000  # Set to a lower number for testing purposes
for i, combination in enumerate(grid_combinations):
    if i >= max_combinations:
        break
    for batch_size in batch_sizes:
        calib_dataset_name, eval_dataset_name, quantize_method, quantize_params, model_name = combination

        # Print current combination details
        print(f"Running combination {i+1}:")
        print(f"  Model Name: {model_name}")
        print(f"  Calibration Dataset: {calib_dataset_name}")
        print(f"  Evaluation Dataset: {eval_dataset_name}")
        print(f"  Quantize Method: {quantize_method}")
        print(f"  Quantize Params: {quantize_params}")
        print(f"  Batch Size: {batch_size}")

        result = run_quantize(
            # Fixed parameters
            device=fixed_params['device'],
            clean_cache=fixed_params['clean_cache'],
            save_quantized_model=fixed_params['save_quantized_model'],
            seed_model=fixed_params['seed_model'],
            seed_dataset=fixed_params['seed_dataset'],
            eval_metrics=fixed_params['eval_metrics'],
            calib_dataset_split=fixed_params['calib_dataset_split'],
            eval_dataset_split=fixed_params['eval_dataset_split'],
            # Grid parameters
            calib_dataset_name=calib_dataset_name,
            eval_dataset_name=eval_dataset_name,
            quantize_method=quantize_method,
            quantize_params=quantize_params,
            model_name=model_name,
            # Random parameters
            batch_size=batch_size,
            dataset_stride=fixed_params['dataset_stride'],
            dataset_seq_length=fixed_params['dataset_seq_length'],
            # Model parameters
            directory_model="",
            directory_dataset="",
            quantized_model_save_path=""
        )

        # Append result with parameter details
        results.append({
            'result': result,
            'parameters': {
                'model_name': model_name,
                'calib_dataset_name': calib_dataset_name,
                'eval_dataset_name': eval_dataset_name,
                'quantize_method': quantize_method,
                'quantize_params': quantize_params,
                'batch_size': batch_size,
                'dataset_stride': fixed_params['dataset_stride'],
                'dataset_seq_length': fixed_params['dataset_seq_length'],
            }
        })

# Do something with the results
print(results)

Running combination 1:
  Model Name: TinyLlama
  Calibration Dataset: WikiText
  Evaluation Dataset: WikiText
  Quantize Method: BNB
  Quantize Params: {'num_bits': 8, 'llm_int8_threshold': 6.0, 'llm_int8_enable_fp32_cpu_offload': False, 'llm_int8_has_fp16_weight': False}
  Batch Size: 1
Context: Load model. Free GPU Memory (GB): 34.8809


Token indices sequence length is longer than the specified maximum sequence length for this model (2874559 > 2048). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (2874559 > 2048). Running this sequence through the model will result in indexing errors


Context: Load data. Free GPU Memory (GB): 34.8809
Model is quantized and saved at "/nfs/students/daro/models/TinyLlama-1.1B-Chat-v1.0-bnb-8bit"
Context: Quantize model. Free GPU Memory (GB): 33.3262
Model in evaluation mode. Device: cuda
Processing batch 0


/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/bitsandbytes/autograd/_functions.py:316: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Perplexity: 7.22
Processing batch 1
Perplexity: 7.22
Processing batch 2
Perplexity: 9.50
Processing batch 3
Perplexity: 11.99

Final Perplexity (PPL): 8.780
Model in evaluation mode. Device: cuda
Processing batch 0
Processing batch 1
Processing batch 2
Processing batch 3
Final Brier Score: 0.0000172327
Context: Evaluate model. Free GPU Memory (GB): 26.5547
Running combination 2:
  Model Name: TinyLlama
  Calibration Dataset: WikiText
  Evaluation Dataset: WikiText
  Quantize Method: BNB
  Quantize Params: {'num_bits': 4, 'bnb_4bit_compute_dtype': torch.bfloat16, 'bnb_4bit_quant_type': 'fp4', 'bnb_4bit_use_double_quant': False}
  Batch Size: 1
Context: Load model. Free GPU Memory (GB): 26.5547


...

/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/bitsandbytes/autograd/_functions.py:316: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Perplexity: 7.22
Processing batch 1
Perplexity: 7.22
Processing batch 2
Perplexity: 9.50
Processing batch 3
Perplexity: 11.99

Final Perplexity (PPL): 8.780
Model in evaluation mode. Device: cuda
Processing batch 0
Processing batch 1
Processing batch 2
Processing batch 3
Final Brier Score: 0.0000172327
Context: Evaluate model. Free GPU Memory (GB): 26.0918
Running combination 6:
  Model Name: TinyLlama
  Calibration Dataset: OpenAssistant
  Evaluation Dataset: WikiText
  Quantize Method: BNB
  Quantize Params: {'num_bits': 4, 'bnb_4bit_compute_dtype': torch.bfloat16, 'bnb_4bit_quant_type': 'fp4', 'bnb_4bit_use_double_quant': False}
  Batch Size: 1
Context: Load model. Free GPU Memory (GB): 26.0918


Token indices sequence length is longer than the specified maximum sequence length for this model (12056829 > 2048). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (2874559 > 2048). Running this sequence through the model will result in indexing errors


Context: Load data. Free GPU Memory (GB): 26.0918
Model is quantized and saved at "/nfs/students/daro/models/TinyLlama-1.1B-Chat-v1.0-bnb-4bit"
Context: Quantize model. Free GPU Memory (GB): 32.8906
Model in evaluation mode. Device: cuda
Processing batch 0
Perplexity: 8.19
Processing batch 1
Perplexity: 8.19
Processing batch 2
Perplexity: 10.41
Processing batch 3
Perplexity: 13.15

Final Perplexity (PPL): 9.791
Model in evaluation mode. Device: cuda
Processing batch 0
Processing batch 1
Processing batch 2
Processing batch 3
Final Brier Score: 0.0000180495
Context: Evaluate model. Free GPU Memory (GB): 27.5918
Running combination 7:
  Model Name: TinyLlama
  Calibration Dataset: OpenAssistant
  Evaluation Dataset: OpenAssistant
  Quantize Method: BNB
  Quantize Params: {'num_bits': 8, 'llm_int8_threshold': 6.0, 'llm_int8_enable_fp32_cpu_offload': False, 'llm_int8_has_fp16_weight': False}
  Batch Size: 1


: 

In [ ]:
print(wikitext_data_module.val_dataset["text"][:100])
print(wikitext_dataloader.dataset.dataset["text"][:100])

print(len(wikitext_data_module.val_dataset["text"]))
print(len(wikitext_dataloader.dataset.dataset["text"]))

In [ ]:
print(oasst_data_module.val_dataset["text"][:100])
print(oasst_dataloader.dataset["text"][:100])

print(len(oasst_data_module.val_dataset["text"]))
print(len(oasst_dataloader.dataset.dataset["text"]))

In [ ]:
print(wikitext_dataloader.dataset)
print(oasst_dataloader.dataset)

In [ ]:
results